# Restaurant Agent Example Without LLM

This notebook shows the restaurant example as code.

The flow is:

`START -> take_order -> send_to_kitchen -> cook_food -> serve_food -> END`

In [ ]:
from typing import Annotated, TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

## State

State is like the order paper. It carries information from one restaurant step to the next step.

In [ ]:
class RestaurantState(TypedDict, total=False):
    messages: Annotated[list, add_messages]
    order: str
    kitchen_status: str
    food_status: str
    served: bool

## Nodes

Each function below is one node. A node does one small job.

In [ ]:
def take_order(state: RestaurantState) -> RestaurantState:
    order = state.get("order", "biryani")
    print("1. Taking order:", order)

    return {
        "order": order,
        "messages": [f"Waiter: Your order for {order} is noted."]
    }


def send_to_kitchen(state: RestaurantState) -> RestaurantState:
    order = state["order"]
    print("2. Sending order to kitchen:", order)

    return {
        "kitchen_status": "order sent to kitchen",
        "messages": [f"Kitchen received the order for {order}."]
    }


def cook_food(state: RestaurantState) -> RestaurantState:
    order = state["order"]
    print("3. Cooking food:", order)

    return {
        "food_status": f"{order} is ready",
        "messages": [f"Chef: The {order} is cooked and ready."]
    }


def serve_food(state: RestaurantState) -> RestaurantState:
    order = state["order"]
    print("4. Serving food:", order)

    return {
        "served": True,
        "messages": [f"Waiter: Here is your {order}. Enjoy your meal!"]
    }

## Build Graph

Edges connect nodes. They decide the order of work.

In [ ]:
restaurant_builder = StateGraph(RestaurantState)

restaurant_builder.add_node("take_order", take_order)
restaurant_builder.add_node("send_to_kitchen", send_to_kitchen)
restaurant_builder.add_node("cook_food", cook_food)
restaurant_builder.add_node("serve_food", serve_food)

restaurant_builder.add_edge(START, "take_order")
restaurant_builder.add_edge("take_order", "send_to_kitchen")
restaurant_builder.add_edge("send_to_kitchen", "cook_food")
restaurant_builder.add_edge("cook_food", "serve_food")
restaurant_builder.add_edge("serve_food", END)

restaurant_graph = restaurant_builder.compile()

In [ ]:
restaurant_graph

## Run Agent

The customer orders biryani. The state moves through every node.

In [ ]:
restaurant_result = restaurant_graph.invoke({
    "order": "biryani",
    "messages": []
})

restaurant_result

In [ ]:
for message in restaurant_result["messages"]:
    print(message.content)